# VARIANCES

This notebook is dedicated at the empirical study of how the variance of the errors vary among different horizons.

In [18]:
from src.data_handler import *
from src.config_files import *
from src.direct_models import *
from src.mheme import *
from src.plot_handler import *

import os

In [19]:
WINDOW = 336
HORIZON = 24

DATA_PATH = '../data'

MODEL_PATH = '../models'

TCN_PATH_CONFIG_LOAD = '../src/config_files/tcn_config.json'
TCN_PATH_SAVE = '../models/tcn'

XGB_PATH_CONFIG_LOAD = '../src/config_files/xgb_config.json'
XGB_PATH_SAVE = '../models/xgb'

In [21]:
X, data = data_loader(data_path = DATA_PATH, dataset = 'electricity')

In [ ]:
kwargs = {"name" : "Electricity", "color" : "green", "title" : "Electricity demand over time", "x_axis" : "Time", "y_axis" : "Electricity demand"}
plot_time_series(X[-500:], **kwargs)

In [24]:
X_slide, y_slide = sliding_window(X, window=WINDOW, horizon=HORIZON, k = 6)
train, val, test = train_validation_test_split(X_slide, y_slide)

In [25]:
#umheme_tcn = UMHEMe.load_model(os.path.join(MODEL_PATH, 'tcn_traffic_2.pkl'))
umheme_tcn = UMHEMe(HORIZON, WINDOW, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD, skip=4)
umheme_tcn.fit(train[0], train[1])

Fitting class : <class 'src.direct_models.TCN'>; horizon : 1


Training TCN: 100%|██████████| 150/150 [00:18<00:00,  8.33it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 5


Training TCN: 100%|██████████| 150/150 [00:19<00:00,  7.75it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 9


...

Training TCN: 100%|██████████| 150/150 [00:18<00:00,  8.23it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 17


Training TCN: 100%|██████████| 150/150 [00:18<00:00,  8.24it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 21


Training TCN: 100%|██████████| 150/150 [00:18<00:00,  8.15it/s]


In [26]:
umheme_tcn.save_model(os.path.join(MODEL_PATH, 'tcn_electrical_5.pkl'))

In [27]:
umheme_tcn.compute_weights(train[0], train[1])

In [15]:
umheme_tcn.visualize_variances(None)
umheme_tcn.visualize_errors(None)
umheme_tcn.visualize_weights(None)

In [30]:
preds = umheme_tcn.whole_predict(test[0])
ens_preds = umheme_tcn.predict(test[0])
for i in range (10):
    pred = {model : preds[model][i] for model in preds}
    plot_multiple_forecast(test[0][i], test[1][i], pred, **kwargs)
    errs = np.array([mse(preds[model][i], test[1][i]) for model in preds])
    avg_mse = np.mean(errs)
    min_mse = np.min(errs)
    max_mse = np.max(errs)
    ensemble_mse = mse(ens_preds[i], test[1][i])
    print(f"min mse : {min_mse}\nmax mse : {max_mse}\naverage mse : {avg_mse}\nensemble mse : {ensemble_mse}")
    

min mse : 12.209582328796387
max mse : 12.951324462890625
average mse : 12.470993995666504
ensemble mse : 12.443425178527832


min mse : 12.134020805358887
max mse : 12.460807800292969
average mse : 12.2617826461792
ensemble mse : 12.230542182922363


min mse : 3.34267520904541
max mse : 3.8932180404663086
average mse : 3.6423397064208984
ensemble mse : 3.45957350730896


min mse : 3.395444869995117
max mse : 3.9191534519195557
average mse : 3.668138265609741
ensemble mse : 3.596677541732788


min mse : 3.267664909362793
max mse : 3.8130569458007812
average mse : 3.5061557292938232
ensemble mse : 3.4655520915985107


min mse : 3.366204261779785
max mse : 3.517580032348633
average mse : 3.4515678882598877
ensemble mse : 3.414926290512085


min mse : 1.6431142091751099
max mse : 2.039930582046509
average mse : 1.8145147562026978
ensemble mse : 1.6467113494873047


min mse : 7.728076934814453
max mse : 9.560673713684082
average mse : 8.686541557312012
ensemble mse : 8.57645320892334


min mse : 8.377470970153809
max mse : 8.936498641967773
average mse : 8.580098152160645
ensemble mse : 8.531025886535645


min mse : 7.938689708709717
max mse : 8.935966491699219
average mse : 8.295509338378906
ensemble mse : 8.23453426361084


In [31]:
preds = umheme_tcn.whole_predict(test[0])
ens_preds = umheme_tcn.predict(test[0])
for i in range (10):
    plot_forecast(test[0][i], test[1][i], ens_preds[i], **kwargs)